<a href="https://colab.research.google.com/github/Pinuig/machine-learning-lab/blob/main/titanic_starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder

TRAIN_PATH = "/content/drive/MyDrive/titanic_starter/train.csv"
TEST_PATH = "/content/drive/MyDrive/titanic_starter/test.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

test_ids = test["PassengerId"].copy()

train["is_train"] = 1
test["is_train"] = 0
test["Survived"] = np.nan
full = pd.concat([train, test], axis=0, ignore_index=True)


def extract_title(name: str) -> str:
    """Pull the honorific (Mr, Mrs, Miss, etc.) out of the Name field."""
    title = name.split(",")[1].split(".")[0].strip()

    rare_titles = {
        "Lady", "Countess", "Capt", "Col", "Don", "Dr", "Major",
        "Rev", "Sir", "Jonkheer", "Dona",
    }
    if title in rare_titles:
        return "Rare"
    if title == "Mlle":
        return "Miss"
    if title == "Ms":
        return "Miss"
    if title == "Mme":
        return "Mrs"
    return title


full["Title"] = full["Name"].apply(extract_title)

full["FamilySize"] = full["SibSp"] + full["Parch"] + 1
full["IsAlone"] = (full["FamilySize"] == 1).astype(int)

full["Deck"] = full["Cabin"].apply(lambda c: c[0] if pd.notnull(c) else "Unknown")

ticket_counts = full["Ticket"].value_counts()
full["TicketGroupSize"] = full["Ticket"].map(ticket_counts)

full["Age"] = full.groupby(["Title", "Pclass"])["Age"].transform(
    lambda x: x.fillna(x.median())
)
full["Age"] = full["Age"].fillna(full["Age"].median())

full["Fare"] = full.groupby("Pclass")["Fare"].transform(
    lambda x: x.fillna(x.median())
)
full["Embarked"] = full["Embarked"].fillna(full["Embarked"].mode()[0])

full["FareBin"] = pd.qcut(full["Fare"], 4, labels=False, duplicates="drop")

full["AgeBin"] = pd.cut(
    full["Age"], bins=[0, 12, 18, 35, 60, 100], labels=False
)

categorical_cols = ["Sex", "Embarked", "Title", "Deck"]
for col in categorical_cols:
    le = LabelEncoder()
    full[col] = le.fit_transform(full[col].astype(str))

feature_cols = [
    "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked",
    "Title", "FamilySize", "IsAlone", "Deck", "TicketGroupSize",
    "FareBin", "AgeBin",
]

train_processed = full[full["is_train"] == 1]
test_processed = full[full["is_train"] == 0]

X = train_processed[feature_cols]
y = train_processed["Survived"].astype(int)
X_test = test_processed[feature_cols]

model = RandomForestClassifier(
    n_estimators=500,
    max_depth=6,
    min_samples_split=4,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")
print(f"CV accuracy: {scores.mean():.4f} (+/- {scores.std():.4f})")

model.fit(X, y)

importances = pd.Series(model.feature_importances_, index=feature_cols)
print("\nTop features:")
print(importances.sort_values(ascending=False).head(10))

predictions = model.predict(X_test).astype(int)

submission = pd.DataFrame({"PassengerId": test_ids, "Survived": predictions})
submission.to_csv("submission.csv", index=False)
print("\nSaved submission.csv")

CV accuracy: 0.8305 (+/- 0.0142)

Top features:
Sex                0.295468
Title              0.156586
Fare               0.104782
Pclass             0.079331
Age                0.067351
Deck               0.062011
TicketGroupSize    0.053271
FamilySize         0.047279
FareBin            0.036913
AgeBin             0.031780
dtype: float64

Saved submission.csv


In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder

TRAIN_PATH = "/content/drive/MyDrive/titanic_starter/train.csv"
TEST_PATH = "/content/drive/MyDrive/titanic_starter/test.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

test_ids = test["PassengerId"].copy()

train["is_train"] = 1
test["is_train"] = 0
test["Survived"] = np.nan
full = pd.concat([train, test], axis=0, ignore_index=True)


def extract_title(name: str) -> str:
    """Pull the honorific (Mr, Mrs, Miss, etc.) out of the Name field."""
    title = name.split(",")[1].split(".")[0].strip()

    rare_titles = {
        "Lady", "Countess", "Capt", "Col", "Don", "Dr", "Major",
        "Rev", "Sir", "Jonkheer", "Dona",
    }
    if title in rare_titles:
        return "Rare"
    if title == "Mlle":
        return "Miss"
    if title == "Ms":
        return "Miss"
    if title == "Mme":
        return "Mrs"
    return title


full["Title"] = full["Name"].apply(extract_title)

full["FamilySize"] = full["SibSp"] + full["Parch"] + 1
full["IsAlone"] = (full["FamilySize"] == 1).astype(int)

full["Deck"] = full["Cabin"].apply(lambda c: c[0] if pd.notnull(c) else "Unknown")

ticket_counts = full["Ticket"].value_counts()
full["TicketGroupSize"] = full["Ticket"].map(ticket_counts)

full["Age"] = full.groupby(["Title", "Pclass"])["Age"].transform(
    lambda x: x.fillna(x.median())
)
full["Age"] = full["Age"].fillna(full["Age"].median())

full["Fare"] = full.groupby("Pclass")["Fare"].transform(
    lambda x: x.fillna(x.median())
)
full["Embarked"] = full["Embarked"].fillna(full["Embarked"].mode()[0])

full["FareBin"] = pd.qcut(full["Fare"], 4, labels=False, duplicates="drop")

full["AgeBin"] = pd.cut(
    full["Age"], bins=[0, 12, 18, 35, 60, 100], labels=False
)

categorical_cols = ["Sex", "Embarked", "Title", "Deck"]
for col in categorical_cols:
    le = LabelEncoder()
    full[col] = le.fit_transform(full[col].astype(str))

feature_cols = [
    "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked",
    "Title", "FamilySize", "IsAlone", "Deck", "TicketGroupSize",
    "FareBin", "AgeBin",
]

train_processed = full[full["is_train"] == 1]
test_processed = full[full["is_train"] == 0]

X = train_processed[feature_cols]
y = train_processed["Survived"].astype(int)
X_test = test_processed[feature_cols]

model = RandomForestClassifier(
    n_estimators=500,
    max_depth=6,
    min_samples_split=4,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")
print(f"CV accuracy: {scores.mean():.4f} (+/- {scores.std():.4f})")

model.fit(X, y)

importances = pd.Series(model.feature_importances_, index=feature_cols)
print("\nTop features:")
print(importances.sort_values(ascending=False).head(10))

predictions = model.predict(X_test).astype(int)

submission = pd.DataFrame({"PassengerId": test_ids, "Survived": predictions})
submission.to_csv("submission.csv", index=False)
print("\nSaved submission.csv")

CV accuracy: 0.8305 (+/- 0.0142)

Top features:
Sex                0.295468
Title              0.156586
Fare               0.104782
Pclass             0.079331
Age                0.067351
Deck               0.062011
TicketGroupSize    0.053271
FamilySize         0.047279
FareBin            0.036913
AgeBin             0.031780
dtype: float64

Saved submission.csv


In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder

TRAIN_PATH = "/content/drive/MyDrive/titanic_starter/train.csv"
TEST_PATH = "/content/drive/MyDrive/titanic_starter/test.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

test_ids = test["PassengerId"].copy()

train["is_train"] = 1
test["is_train"] = 0
test["Survived"] = np.nan
full = pd.concat([train, test], axis=0, ignore_index=True)


def extract_title(name: str) -> str:
    """Pull the honorific (Mr, Mrs, Miss, etc.) out of the Name field."""
    title = name.split(",")[1].split(".")[0].strip()

    rare_titles = {
        "Lady", "Countess", "Capt", "Col", "Don", "Dr", "Major",
        "Rev", "Sir", "Jonkheer", "Dona",
    }
    if title in rare_titles:
        return "Rare"
    if title == "Mlle":
        return "Miss"
    if title == "Ms":
        return "Miss"
    if title == "Mme":
        return "Mrs"
    return title


full["Title"] = full["Name"].apply(extract_title)

full["FamilySize"] = full["SibSp"] + full["Parch"] + 1
full["IsAlone"] = (full["FamilySize"] == 1).astype(int)

full["Deck"] = full["Cabin"].apply(lambda c: c[0] if pd.notnull(c) else "Unknown")

ticket_counts = full["Ticket"].value_counts()
full["TicketGroupSize"] = full["Ticket"].map(ticket_counts)

full["Age"] = full.groupby(["Title", "Pclass"])["Age"].transform(
    lambda x: x.fillna(x.median())
)
full["Age"] = full["Age"].fillna(full["Age"].median())

full["Fare"] = full.groupby("Pclass")["Fare"].transform(
    lambda x: x.fillna(x.median())
)
full["Embarked"] = full["Embarked"].fillna(full["Embarked"].mode()[0])

full["FareBin"] = pd.qcut(full["Fare"], 4, labels=False, duplicates="drop")

full["AgeBin"] = pd.cut(
    full["Age"], bins=[0, 12, 18, 35, 60, 100], labels=False
)

categorical_cols = ["Sex", "Embarked", "Title", "Deck"]
for col in categorical_cols:
    le = LabelEncoder()
    full[col] = le.fit_transform(full[col].astype(str))

feature_cols = [
    "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked",
    "Title", "FamilySize", "IsAlone", "Deck", "TicketGroupSize",
    "FareBin", "AgeBin",
]

train_processed = full[full["is_train"] == 1]
test_processed = full[full["is_train"] == 0]

X = train_processed[feature_cols]
y = train_processed["Survived"].astype(int)
X_test = test_processed[feature_cols]

model = RandomForestClassifier(
    n_estimators=500,
    max_depth=6,
    min_samples_split=4,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")
print(f"CV accuracy: {scores.mean():.4f} (+/- {scores.std():.4f})")

model.fit(X, y)

importances = pd.Series(model.feature_importances_, index=feature_cols)
print("\nTop features:")
print(importances.sort_values(ascending=False).head(10))

predictions = model.predict(X_test).astype(int)

submission = pd.DataFrame({"PassengerId": test_ids, "Survived": predictions})
submission.to_csv("submission.csv", index=False)
print("\nSaved submission.csv")

CV accuracy: 0.8305 (+/- 0.0142)

Top features:
Sex                0.295468
Title              0.156586
Fare               0.104782
Pclass             0.079331
Age                0.067351
Deck               0.062011
TicketGroupSize    0.053271
FamilySize         0.047279
FareBin            0.036913
AgeBin             0.031780
dtype: float64

Saved submission.csv
